In [22]:
import pyomo.environ as pyo
from pyomo.gdp import Disjunction, Disjunct
import json
import os

In [23]:
json ={
    "jobs": [1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15],
    "processing_time": {
        "1": 34, "2": 38, "3": 39, "4": 20, "5": 47, "6": 47, "7": 14, "8": 13,
        "9": 28, "10": 20, "11": 32, "12": 25, "13": 16, "14": 35, "15": 11
    },
    "release_time": {
        "1": 64, "2": 195, "3": 200, "4": 192, "5": 135, "6": 189, "7": 82, "8": 104,
        "9": 123, "10": 29, "11": 43, "12": 49, "13": 156, "14": 11, "15": 52
    },
    "due_time": {
        "1": 302, "2": 461, "3": 473, "4": 332, "5": 464, "6": 518, "7": 180, "8": 195,
        "9": 319, "10": 169, "11": 267, "12": 224, "13": 268, "14": 256, "15": 129
    }
}

In [58]:
m = pyo.ConcreteModel()

m.I = pyo.Set(initialize=json["jobs"])
m.p = pyo.Param(m.I, initialize={int(k): v for k, v in json["processing_time"].items()})
m.r = pyo.Param(m.I, initialize={int(k): v for k, v in json["release_time"].items()})
m.d = pyo.Param(m.I, initialize={int(k): v for k, v in json["due_time"].items()})

# Define the bounds for start times: 
# Each job must start no earlier than its release time and no later than its due date minus processing time.
def x_bounds_rule(m, i):
    return (m.r[i], m.d[i] - m.p[i])
m.x = pyo.Var(m.I, bounds=x_bounds_rule)

# Introduce binary variables:
# y_first[i] = 1 if job i is chosen as the first job.
# y_last[i]  = 1 if job i is chosen as the last job.
m.y_first = pyo.Var(m.I, domain=pyo.Binary)
m.y_last  = pyo.Var(m.I, domain=pyo.Binary)
m.y = pyo.Var(m.I, m.I, domain=pyo.Binary)

# Compute bounds for makespan:
lower_bound_makespan = min(json["release_time"][str(i)] + json["processing_time"][str(i)] for i in json["jobs"])
upper_bound_makespan = max(json["due_time"][str(i)] for i in json["jobs"])
m.makespan = pyo.Var(bounds=(lower_bound_makespan, upper_bound_makespan))


def M_rule(m, i, j):
    if i == j:
        return pyo.Param.Skip
    else:
        return m.d[i] - m.r[j]
m.M = pyo.Param(m.I, m.I, initialize=M_rule, within=pyo.Any)

def immediate_precedence_rule(m, i, j):
    if i == j:
        return pyo.Constraint.Skip
    else:
        return m.x[i] + m.p[i] <= m.x[j] + m.M[i, j] * (1 - m.y[i, j])
m.immediate_precedence = pyo.Constraint(m.I, m.I, rule=immediate_precedence_rule)

def first_job_constraint_rule(m, i, j):
    if i == j:
        return pyo.Constraint.Skip
    else:
        return m.x[i] + m.p[i] <= m.x[j] + m.M[i, j] * (1 - m.y_first[i])
m.first_job_constraint = pyo.Constraint(m.I, m.I, rule=first_job_constraint_rule)

def last_job_constraint_rule(m, i, j):
    if i == j:
        return pyo.Constraint.Skip
    else:
        return m.x[j] + m.p[j] <= m.x[i] + m.M[i, j] * (1 - m.y_last[j])
m.last_job_constraint = pyo.Constraint(m.I, m.I, rule=last_job_constraint_rule)

def predecessor_assignment_rule(m, i):
    return sum(m.y[i,j] for j in m.I if j != i) + m.y_first[i] == 1
m.predecessor_assignment = pyo.Constraint(m.I, rule=predecessor_assignment_rule)

def successor_assignment_rule(m, i):
    return m.y_last[i] + sum(m.y[j,i] for j in m.I if j != i) == 1
m.successor_assignment = pyo.Constraint(m.I, rule=successor_assignment_rule)

def first_job_rule(m):
    return sum(m.y_first[i] for i in m.I) == 1
m.first_job = pyo.Constraint(rule=first_job_rule)

def last_job_rule(m):
    return sum(m.y_last[i] for i in m.I) == 1
m.last_job = pyo.Constraint(rule=last_job_rule)

def demorgan_rule(m, i):
    return m.y_first[i] + m.y_last[i] <= 1
m.demorgan = pyo.Constraint(m.I, rule=demorgan_rule)
m.demorgan.pprint()

def makespan_constraint(m, i):
    return m.x[i] + m.p[i] <= m.makespan
m.makespan_constraint = pyo.Constraint(m.I, rule=makespan_constraint)

# Define objective: minimize makespan
m.obj = pyo.Objective(expr=m.makespan, sense=pyo.minimize)

demorgan : Size=15, Index=I, Active=True
    Key : Lower : Body                     : Upper : Active
      1 :  -Inf :   y_first[1] + y_last[1] :   1.0 :   True
      2 :  -Inf :   y_first[2] + y_last[2] :   1.0 :   True
      3 :  -Inf :   y_first[3] + y_last[3] :   1.0 :   True
      4 :  -Inf :   y_first[4] + y_last[4] :   1.0 :   True
      5 :  -Inf :   y_first[5] + y_last[5] :   1.0 :   True
      6 :  -Inf :   y_first[6] + y_last[6] :   1.0 :   True
      7 :  -Inf :   y_first[7] + y_last[7] :   1.0 :   True
      8 :  -Inf :   y_first[8] + y_last[8] :   1.0 :   True
      9 :  -Inf :   y_first[9] + y_last[9] :   1.0 :   True
     10 :  -Inf : y_first[10] + y_last[10] :   1.0 :   True
     11 :  -Inf : y_first[11] + y_last[11] :   1.0 :   True
     12 :  -Inf : y_first[12] + y_last[12] :   1.0 :   True
     13 :  -Inf : y_first[13] + y_last[13] :   1.0 :   True
     14 :  -Inf : y_first[14] + y_last[14] :   1.0 :   True
     15 :  -Inf : y_first[15] + y_last[15] :   1.0 :   True

In [41]:
m = pyo.ConcreteModel()

m.I = pyo.Set(initialize=json["jobs"])
m.p = pyo.Param(m.I, initialize={int(k): v for k, v in json["processing_time"].items()})
m.r = pyo.Param(m.I, initialize={int(k): v for k, v in json["release_time"].items()})
m.d = pyo.Param(m.I, initialize={int(k): v for k, v in json["due_time"].items()})

# Immediate precedence Concepts
m.x = pyo.Var(m.I, within=pyo.NonNegativeReals)

# first job disjuncts
def first_job_disjunct_rule(disjunct, i):
    m = disjunct.model()
    disjunct.cons=pyo.ConstraintList()
    # xi+ pi <= xj for all i not equal to j
    for j in m.I:
        if i != j:
            disjunct.cons.add(m.x[i] + m.p[i] <= m.x[j])
m.first_job_disjunct = Disjunct(m.I, rule=first_job_disjunct_rule)

# last job disjuncts
def last_job_disjunct_rule(disjunct, i):
    m = disjunct.model()
    disjunct.cons=pyo.ConstraintList()
    # xj + pj <= xi for all i not
    for j in m.I:
        if i != j:
            disjunct.cons.add(m.x[j] + m.p[j] <= m.x[i])
m.last_job_disjunct = Disjunct(m.I, rule=last_job_disjunct_rule)

def first_job_disjunction_rule(m):
# Return a list of first-job disjuncts, one per job
        return [m.first_job_disjunct[i] for i in m.I]
m.first_job_disjunction = Disjunction(rule=first_job_disjunction_rule)

def last_job_disjunction_rule(m):
# Return a list of last-job disjuncts, one per job
    return [m.last_job_disjunct[i] for i in m.I]
m.last_job_disjunction = Disjunction(rule=last_job_disjunction_rule)

# Logic Expression
def logic_expression_rule(m, i):
    return pyo.lnot(pyo.land(m.first_job_disjunct[i].indicator_var, m.last_job_disjunct[i].indicator_var))
m.logic_expression = pyo.LogicalConstraint(m.I, rule=logic_expression_rule)

# Immediate precedence disjuncts
def immediate_precedence_disjunct_rule(disjunct, i, j):
    m = disjunct.model()
    if i == j:
        disjunct.deactivate() # Deactivate the disjunct if i == j
    else:
        disjunct.cons = pyo.Constraint(expr=m.x[i] + m.p[i] <= m.x[j])
m.immediate_precedence_disjunct = Disjunct(m.I, m.I, rule=immediate_precedence_disjunct_rule)

# Successor Disjunction
def successor_disjunction_rule(m, i):
    return [m.immediate_precedence_disjunct[i, j] for j in m.I if j != i] + [m.last_job_disjunct[i]]
m.SuccessorDisjunction = Disjunction(m.I, rule=successor_disjunction_rule)

# Predecessor Disjunction
def predecessor_disjunction_rule(m, i):
    return [m.immediate_precedence_disjunct[j, i] for j in m.I if j != i] + [m.first_job_disjunct[i]]
m.PredecessorDisjunction = Disjunction(m.I, rule=predecessor_disjunction_rule)